# 1225. Report Contiguous Dates

## Problem
We need to generate a report of **continuous intervals of days** between `2019-01-01` and `2019-12-31`.  
- Each day has either a **failed** or **succeeded** task.  
- The report should group consecutive days with the same state into intervals.  
- Output columns: `period_state`, `start_date`, `end_date`.  
- Order results by `start_date`.

---

## Schema

### Table: Failed
| Column Name | Type | Description                  |
|-------------|------|------------------------------|
| fail_date   | DATE | Primary key, failed task day |

### Table: Succeeded
| Column Name   | Type | Description                     |
|---------------|------|---------------------------------|
| success_date  | DATE | Primary key, succeeded task day |

---

## Sample Data

### Failed
| fail_date   |
|-------------|
| 2018-12-28  |
| 2018-12-29  |
| 2019-01-04  |
| 2019-01-05  |

### Succeeded
| success_date |
|--------------|
| 2018-12-30   |
| 2018-12-31   |
| 2019-01-01   |
| 2019-01-02   |
| 2019-01-03   |
| 2019-01-06   |

---

## Expected Result
| period_state | start_date | end_date   |
|--------------|------------|------------|
| succeeded    | 2019-01-01 | 2019-01-03 |
| failed       | 2019-01-04 | 2019-01-05 |
| succeeded    | 2019-01-06 | 2019-01-06 |

---

## PySpark Code: Create DataFrames and Temp Views

```python


In [0]:
from pyspark.sql.types import StructType, StructField, DateType
from datetime import date

# Schema for Failed
failed_schema = StructType([
    StructField("fail_date", DateType(), False)
])

# Schema for Succeeded
succeeded_schema = StructType([
    StructField("success_date", DateType(), False)
])

# Data for Failed
failed_data = [
    (date(2018,12,28),),
    (date(2018,12,29),),
    (date(2019,1,4),),
    (date(2019,1,5),)
]

# Data for Succeeded
succeeded_data = [
    (date(2018,12,30),),
    (date(2018,12,31),),
    (date(2019,1,1),),
    (date(2019,1,2),),
    (date(2019,1,3),),
    (date(2019,1,6),)
]

# Create DataFrames
failed_df = spark.createDataFrame(failed_data, failed_schema)
succeeded_df = spark.createDataFrame(succeeded_data, succeeded_schema)

# Register Temp Views
failed_df.createOrReplaceTempView("Failed")
succeeded_df.createOrReplaceTempView("Succeeded")

# Quick check
failed_df.show()
succeeded_df.show()


In [0]:
%sql

 Select success_date as event_date  , 'Success' as status from Succeeded

In [0]:
%sql
with cte as (
    Select fail_date as event_date , 'Fail' as status from Failed WHERE fail_date BETWEEN  '2019-01-01' AND   '2019-12-31'
    union all
    Select success_date as event_date  , 'Success' as status from Succeeded WHERE success_date BETWEEN  '2019-01-01' AND   '2019-12-31'
    order by event_date asc
)
,cte_2
(
  Select 
  (row_number()over(partition by  status order by event_date asc)) as rn
  
  ,* from cte
  order by event_date asc
) , CTE3(
select cast(date_add(day , - rn , event_date) as date) as start_date 

,* 


from cte_2), CTE4 AS(
SELECT  
STATUS AS PERIOD_STATE
, MIN(EVENT_DATE)OVER(PARTITION BY STATUS   , START_DATE) AS START_DATE
, MAX(EVENT_DATE)OVER(PARTITION BY STATUS , START_DATE) AS END_DATE

 FROM CTE3 ORDER BY start_date )
 SELECT DISTINCT * FROM CTE4
-- WHERE START_DATE >=  '2019-01-01' --AND END_DATE >= '2019-12-31'
  ORDER BY START_DATE

# Reflection on Problem-Solving: Contiguous Dates Challenge

## Why This Was Unique
- This was a **very unique challenge** that required careful use of window functions and grouping logic.
- It took me a long time to solve, and I had to refer to a solution before fully understanding it.

---

## Mistake I Made
- I used the **wrong column** in the `MIN()` window function.
- Example of the mistake:
  ```sql
  MIN(EVENT_DATE) OVER (PARTITION BY STATUS, START_DATE) AS START_DATE



## Correct Thinking
- The key idea is to:
  1. Combine all failed and succeeded dates into one timeline (`cte`).
  2. Use `ROW_NUMBER()` to assign sequence numbers per status (`cte_2`).
  3. Calculate a **derived start_date** by subtracting the row number from the event date (`cte_3`).
     - This trick groups consecutive dates with the same status into the same interval.
  4. Use window functions (`MIN` and `MAX`) over these groups to find the actual start and end of each contiguous block (`cte_4`).
  5. Select distinct intervals to produce the final report.

---

## Step-by-Step Code Explanation

### 1. `cte`
```sql
SELECT fail_date AS event_date, 'Fail' AS status
FROM Failed
WHERE fail_date BETWEEN '2019-01-01' AND '2019-12-31'
UNION ALL
SELECT success_date AS event_date, 'Success' AS status
FROM Succeeded
WHERE success_date BETWEEN '2019-01-01' AND '2019-12-31'
ORDER BY event_date ASC;
```
- Combines failed and succeeded dates into one dataset.
- Adds a `status` column to distinguish them.

---

### 2. `cte_2`
```sql
SELECT 
  ROW_NUMBER() OVER (PARTITION BY status ORDER BY event_date ASC) AS rn,
  *
FROM cte
ORDER BY event_date ASC;
```
- Assigns a sequential number (`rn`) to each event per status.
- Helps in detecting continuity.

---

### 3. `cte_3`
```sql
SELECT 
  CAST(date_add(day, -rn, event_date) AS date) AS start_date,
  *
FROM cte_2;
```
- Subtracts the row number from the event date.
- Consecutive dates with the same status will yield the same `start_date`.
- This is the **grouping trick** that identifies contiguous blocks.

---

### 4. `cte_4`
```sql
SELECT  
  status AS period_state,
  MIN(event_date) OVER (PARTITION BY status, start_date) AS start_date,
  MAX(event_date) OVER (PARTITION BY status, start_date) AS end_date
FROM cte_3
ORDER BY start_date;
```
- Groups by `status` and the derived `start_date`.
- Finds the minimum and maximum event dates in each group.
- Produces the start and end of each contiguous interval.

---

### 5. Final Output
```sql
SELECT DISTINCT * 
FROM cte_4
ORDER BY start_date;
```
- Removes duplicates.
- Orders intervals by start date.

---

## Key Learning
- Always ensure the **correct column** is used in window functions.
- `EVENT_DATE` is the actual timeline column; `START_DATE` is derived for grouping.
- Misusing `START_DATE` in `MIN()` caused misalignment with expected results.
- The grouping trick `(event_date - row_number)` is powerful for detecting contiguous ranges.

---
```
 

In [0]:
%sql
with cte as (
    Select fail_date as event_date , 'Fail' as status from Failed WHERE fail_date BETWEEN  '2019-01-01' AND   '2019-12-31'
    union all
    Select success_date as event_date  , 'Success' as status from Succeeded WHERE success_date BETWEEN  '2019-01-01' AND   '2019-12-31'
    order by event_date asc
)
,cte_2
(
  Select 
  (row_number()over(partition by  status order by event_date asc)) as rn
  
  ,* from cte
  order by event_date asc
) , CTE3(
select cast(date_add(day , - rn , event_date) as date) as start_date 

,* 


from cte_2)
SELECT  DISTINCT 
STATUS AS PERIOD_STATE
, MIN(EVENT_DATE)OVER(PARTITION BY STATUS   , START_DATE) AS START_DATE
, MAX(EVENT_DATE)OVER(PARTITION BY STATUS , START_DATE) AS END_DATE

 FROM CTE3 ORDER BY start_date 
